# Edureka Demo: Modular RAG Pipeline with Hugging Face & FAISS

This notebook demonstrates a **Retrieval-Augmented Generation (RAG)** pipeline step-by-step.
RAG enhances Large Language Models (LLMs) by retrieving context from external files (like `.txt` or `.pdf`) and using it to answer user queries.

The modular pipeline consists of:
1. **Data Ingestion (loader)**: Reads raw text or PDF documents.
2. **Text Splitting (splitter)**: Chunks documents into overlapping segments.
3. **Embeddings (embedder)**: Converts text segments into dense vector coordinates using Hugging Face.
4. **Vector Store (vector_db)**: Stores vectors and retrieves relevant segments using FAISS.
5. **Interactive Query (main)**: Performs similarity queries based on distance scores.

---
## Setup & Prerequisites

Before running this notebook, ensure you have the required libraries installed.
You can run the following cell to verify all imports.


In [ ]:
import os
import shutil
from typing import List, Tuple

# Verify dependencies
try:
    import langchain
    import langchain_community
    import sentence_transformers
    import faiss
    print("[SUCCESS] All dependencies are successfully installed!")
except ImportError as e:
    print(f"[ERROR] Missing dependency error: {e}")
    print("Please run: pip install -r requirements.txt")


## Step 1: Data Ingestion & Document Loaders

The ingestion step loads raw text files or PDFs and converts them into standard LangChain `Document` objects. Each document contains:
- `page_content`: The actual text.
- `metadata`: Source information (like filename or page number).

Here is the implementation of `load_document` from `loader.py`:


In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader

def load_document(file_path: str) -> List[Document]:
    """
    Loads a document from the specified file path based on its extension.
    Supports .txt and .pdf formats.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file at {file_path} was not found.")
        
    ext = os.path.splitext(file_path)[1].lower()
    print(f"[Loader] Loading file: {file_path} (Type: {ext})")
    
    if ext == ".txt":
        loader = TextLoader(file_path, encoding="utf-8")
        return loader.load()
    elif ext == ".pdf":
        loader = PyPDFLoader(file_path)
        return loader.load()
    else:
        raise ValueError(f"Unsupported file format '{ext}'. Only .txt and .pdf are supported.")


In [ ]:
# Let's test the Loader using the provided sample_document.txt
SAMPLE_DOC_PATH = "sample_document.txt"

if os.path.exists(SAMPLE_DOC_PATH):
    raw_docs = load_document(SAMPLE_DOC_PATH)
    print(f"\nSuccessfully loaded {len(raw_docs)} document(s).")
    print(f"Total Character Length: {len(raw_docs[0].page_content)} characters")
    print(f"Metadata: {raw_docs[0].metadata}")
    print("\n--- Document Header Preview ---")
    print("\n".join(raw_docs[0].page_content.splitlines()[:5]))
else:
    print(f"[WARNING] Please make sure '{SAMPLE_DOC_PATH}' exists in the current directory.")


## Step 2: Text Splitting & Chunking

LLMs and embedding models have input size limitations (context windows). To handle large documents, we split the document into smaller, manageable text segments (chunks).
- **Chunk Size**: The maximum number of characters in a chunk.
- **Chunk Overlap**: The number of characters shared between consecutive chunks. This prevents losing context at the boundaries.

We use LangChain's `RecursiveCharacterTextSplitter`, which splits using logical separators (`\n\n`, `\n`, ` `, and `""`) to keep paragraphs and sentences intact as much as possible.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(
    documents: List[Document], 
    chunk_size: int = 500, 
    chunk_overlap: int = 50
) -> List[Document]:
    """
    Splits a list of Documents into smaller overlapping chunks.
    """
    print(f"[Splitter] Splitting {len(documents)} document(s) with chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False
    )
    
    chunks = splitter.split_documents(documents)
    print(f"[Splitter] Created {len(chunks)} text chunks.")
    return chunks


In [ ]:
# Let's chunk the sample document with chunk_size=400 and overlap=50
chunk_size = 400
chunk_overlap = 50

if 'raw_docs' in locals():
    chunks = split_documents(raw_docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    print(f"\nCreated {len(chunks)} chunks from the document.")
    
    # Preview the third chunk
    print(f"\n--- Previewing Chunk #3 (Length: {len(chunks[2].page_content)} chars) ---")
    print(chunks[2].page_content)
    print("-" * 50)
else:
    print("[WARNING] Please run the ingestion code cell first.")


## Step 3: Text Embeddings (sentence-transformers)

An embedding model converts a text string into a list of floating-point numbers (a dense vector).
- These vectors capture the **semantic meaning** of the text.
- Standard models like `all-MiniLM-L6-v2` output **384-dimensional** vectors.
- Words or sentences with similar meanings (e.g., "king" and "queen", or "RAG" and "Retrieval-Augmented Generation") will be closer in vector space.

We use LangChain's `HuggingFaceEmbeddings` wrapper to load the model and generate embeddings locally.


In [ ]:
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

def get_embedding_model(model_name: str = "sentence-transformers/all-MiniLM-L6-v2") -> HuggingFaceEmbeddings:
    """
    Initializes and returns a local HuggingFaceEmbeddings model.
    This model runs entirely locally and downloads the weights if they are not already cached.
    """
    print(f"[Embedder] Loading Hugging Face Embedding Model: {model_name}")
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True} # Normalizes for easy cosine similarity calculations
    )
    print("[Embedder] Embedding model loaded successfully.")
    return embeddings


In [ ]:
# Load the embedding model
embeddings = get_embedding_model()

# Vectorize a sample text
sample_text = "What is Retrieval-Augmented Generation?"
sample_vector = embeddings.embed_query(sample_text)

print(f"\nText: '{sample_text}'")
print(f"Generated Vector Dimension: {len(sample_vector)}")
print(f"Preview of first 5 values: {sample_vector[:5]}")


## Step 4: Vector Database Integration (FAISS)

Once we have embeddings for all text chunks, we index them in a vector database.
- **FAISS (Facebook AI Similarity Search)** is a highly optimized local library for vector search.
- When searching, the database compares the user's query vector against all chunk vectors in the database to find the closest matches.
- We will define functions to **create**, **save**, and **load** the FAISS index to/from the local disk.


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings

def create_vector_store(documents: List[Document], embedding_model: Embeddings) -> FAISS:
    """Creates a FAISS vector database from document chunks and an embedding model."""
    print(f"[VectorDB] Creating FAISS index for {len(documents)} document chunks...")
    db = FAISS.from_documents(documents, embedding_model)
    print("[VectorDB] FAISS index created successfully.")
    return db

def save_vector_store(db: FAISS, folder_path: str):
    """Saves the FAISS index files locally to disk."""
    print(f"[VectorDB] Saving FAISS index locally to folder: {folder_path}")
    db.save_local(folder_path)
    print("[VectorDB] FAISS index saved successfully.")

def load_vector_store(folder_path: str, embedding_model: Embeddings) -> FAISS:
    """Loads a FAISS index from local disk storage."""
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"FAISS index folder not found at: {folder_path}")
    print(f"[VectorDB] Loading FAISS index from folder: {folder_path}")
    db = FAISS.load_local(
        folder_path, 
        embedding_model, 
        allow_dangerous_deserialization=True # Required by LangChain to load metadata pickle files
    )
    print("[VectorDB] FAISS index loaded successfully.")
    return db


In [ ]:
FAISS_STORE_PATH = "faiss_index_store_notebook"

# Remove any existing database files from previous runs
if os.path.exists(FAISS_STORE_PATH):
    shutil.rmtree(FAISS_STORE_PATH)

if 'chunks' in locals() and 'embeddings' in locals():
    # Create index
    vector_db = create_vector_store(chunks, embeddings)
    
    # Save to local disk
    save_vector_store(vector_db, FAISS_STORE_PATH)
else:
    print("[WARNING] Please make sure chunks and embeddings have been initialized.")


## Step 5: Document Retrieval & Querying (Semantic Search)

Now that our index is populated and persisted, we can retrieve relevant document chunks based on a search query.
- When you query FAISS using `similarity_search_with_score`, it returns matching `Document` chunks along with their **L2 (Euclidean) distance** scores.
- **L2 Distance Score**:
  - A **lower score** means the vectors are closer together (meaning they are highly semantically similar).
  - A **higher score** means the vectors are farther apart (less semantically similar).


In [ ]:
def retrieve_similar_documents(
    db: FAISS, 
    query: str, 
    k: int = 3
) -> List[Tuple[Document, float]]:
    """Queries the FAISS vector database to retrieve top K most similar chunks."""
    print(f"[VectorDB] Searching FAISS database for query: '{query}' (Retrieving top {k})")
    results = db.similarity_search_with_score(query, k=k)
    return results


In [ ]:
# Load vector store from disk
if 'embeddings' in locals():
    loaded_db = load_vector_store(FAISS_STORE_PATH, embeddings)
    
    # Query examples
    queries = [
        "What is Retrieval-Augmented Generation?",
        "What is a Vector Database and FAISS?",
        "What are some popular LLM models?"
    ]
    
    for query in queries:
        print("\n" + "="*80)
        print(f"SEARCH QUERY: '{query}'")
        print("="*80)
        
        results = retrieve_similar_documents(loaded_db, query, k=2)
        
        for i, (doc, score) in enumerate(results):
            print(f"\nResult #{i+1} [Similarity Score (L2 Distance): {score:.4f}]")
            print(f"Source: {doc.metadata.get('source', 'Unknown')}")
            print(f"Content:")
            print("-" * 50)
            print(doc.page_content)
            print("-" * 50)
else:
    print("[WARNING] Please initialize embeddings first.")


In [ ]:
# Clean up our temporary FAISS index folder to keep the directory clean
if os.path.exists(FAISS_STORE_PATH):
    shutil.rmtree(FAISS_STORE_PATH)
    print("Clean up complete! Temporary index deleted.")


## Summary & Conclusion

In this notebook, you have successfully built a modular **RAG (Retrieval-Augmented Generation)** ingestion and retrieval pipeline from scratch:
1. **Document Loading**: Ingested raw files into standard LangChain Document objects.
2. **Document Splitting**: Partitioned long text into overlapping chunks using `RecursiveCharacterTextSplitter`.
3. **Text Embeddings**: Loaded a Hugging Face model (`all-MiniLM-L6-v2`) locally to convert chunks into dense vector representations.
4. **Vector Database**: Loaded chunk vectors into **FAISS** for fast indexing and disk storage.
5. **Semantic Retrieval**: Queried the FAISS database using natural language queries to retrieve highly relevant chunks based on semantic distance rather than simple keyword matches.

This modular structure is exactly what runs behind the **Streamlit Web Dashboard** (`app.py`) and the **Command Line Interface** (`main.py`) in this folder. You can now experiment by altering `chunk_size` or testing different search queries!

---
*Created for Edureka Module 3 - RAG Demonstration*
